# TMDb Movie Catalog Analytics: Data Quality, SQL & Statistical EDA

**Author:** Senior Data Analyst & Analytics Engineer  
**Dataset:** `mymoviedb.csv` (TMDb 9,827 Movies Export)  
**Database:** `movies_analytics.db` (SQLite)  

This notebook presents an end-to-end analytics workflow:
1. **Data Quality Auditing & Preprocessing:** Resolving quantile discretization bugs and multi-label normalization.
2. **Dual-Denominator Genre Analysis:** Calculating Tag Share vs. Title Penetration.
3. **Relational SQL Analytics:** Executing CTEs, window functions (`LAG`, `DENSE_RANK`), and self-joins in SQLite.
4. **Statistical Testing:** Spearman rank correlations and Kruskal-Wallis distribution tests.

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries imported successfully.')

## 1. Load Cleaned Dataset & Relational SQLite Database

In [ ]:
base_dir = os.path.abspath('..') if os.path.exists('../data') else '.'
movies_path = os.path.join(base_dir, 'data', 'processed', 'movies_cleaned.csv')
genres_path = os.path.join(base_dir, 'data', 'processed', 'movie_genres.csv')
db_path = os.path.join(base_dir, 'data', 'processed', 'movies_analytics.db')

df_movies = pd.read_csv(movies_path)
df_genres = pd.read_csv(genres_path)
conn = sqlite3.connect(db_path)

print(f'Movies Table: {len(df_movies)} rows, {len(df_movies.columns)} columns')
print(f'Genre Bridge Table: {len(df_genres)} rows, {df_genres["genre"].nunique()} unique genres')
print(f'Unrated Preserved Movies: {(df_movies["Rating_Status"] == "Unrated").sum()} titles')

## 2. Multi-Label Genre Analysis (Dual Denominators)

In [ ]:
genre_counts = df_genres['genre'].value_counts().reset_index()
genre_counts.columns = ['genre', 'occurrence_count']

# Dual Denominators
total_tags = len(df_genres)
total_titles = len(df_movies)

genre_counts['tag_share_pct'] = (genre_counts['occurrence_count'] / total_tags) * 100.0
genre_counts['title_penetration_pct'] = (genre_counts['occurrence_count'] / total_titles) * 100.0

print('--- Top 5 Genres ---')
print(genre_counts.head(5).to_string(index=False))

# Plot Title Penetration
plt.figure(figsize=(10, 6))
sns.barplot(data=genre_counts, x='title_penetration_pct', y='genre', color='#3498db')
plt.title('Genre Title Penetration Rate (% of Catalog Titles)', fontsize=14, fontweight='bold')
plt.xlabel('Title Penetration (%)')
plt.ylabel('Genre')
plt.tight_layout()
plt.show()

## 3. SQL Analytics (Window Functions & YoY Growth in SQLite)

In [ ]:
query_yoy = '''
WITH yearly AS (
    SELECT Release_Year, COUNT(*) AS count
    FROM movies
    WHERE Release_Year BETWEEN 2014 AND 2022
    GROUP BY Release_Year
)
SELECT 
    Release_Year,
    count AS releases,
    LAG(count) OVER (ORDER BY Release_Year) AS prev_year,
    count - LAG(count) OVER (ORDER BY Release_Year) AS net_change,
    ROUND(((count - LAG(count) OVER (ORDER BY Release_Year)) * 100.0) / LAG(count) OVER (ORDER BY Release_Year), 2) AS yoy_growth_pct
FROM yearly
WHERE Release_Year >= 2015;
'''
df_yoy = pd.read_sql_query(query_yoy, conn)
print(df_yoy.to_string(index=False))

# Visualizing Annual Releases & 2021 Peak
plt.figure(figsize=(9, 4.5))
sns.barplot(data=df_yoy, x='Release_Year', y='releases', palette='Blues_d')
plt.title('Annual Movie Releases (2015–2022) with 2020 Shock & 2021 Surge', fontsize=13, fontweight='bold')
plt.xlabel('Release Year')
plt.ylabel('Releases')
plt.tight_layout()
plt.show()

## 4. SQL Self-Join: Genre Co-Occurrence Matrix

In [ ]:
query_cooccur = '''
WITH pairs AS (
    SELECT g1.genre AS genre_a, g2.genre AS genre_b, COUNT(*) AS pair_frequency
    FROM movie_genres g1
    JOIN movie_genres g2 ON g1.movie_id = g2.movie_id AND g1.genre < g2.genre
    GROUP BY g1.genre, g2.genre
)
SELECT genre_a, genre_b, pair_frequency,
       ROUND((pair_frequency * 100.0) / (SELECT COUNT(*) FROM movies), 2) AS title_pct
FROM pairs
ORDER BY pair_frequency DESC
LIMIT 8;
'''
df_cooccur = pd.read_sql_query(query_cooccur, conn)
print(df_cooccur.to_string(index=False))

## 5. Statistical Testing: Spearman Correlation & Kruskal-Wallis

In [ ]:
rated = df_movies[df_movies['Rating_Status'] == 'Rated']
rho, p_val = stats.spearmanr(rated['Popularity'], rated['Vote_Average'])
print(f'Spearman Correlation (Popularity vs. Rating): rho = {rho:.4f}, p = {p_val:.2e}')

# Kruskal-Wallis Test across Top 5 Genres (min 50 votes)
merged = df_genres.merge(rated[rated['Vote_Count'] >= 50], on='movie_id')
top5 = ['Drama', 'Comedy', 'Action', 'Thriller', 'Adventure']
groups = [merged[merged['genre'] == g]['Vote_Average'].values for g in top5]
h_stat, kw_p = stats.kruskal(*groups)
print(f'Kruskal-Wallis Test on Top 5 Genres (H-stat): {h_stat:.2f}, p-value: {kw_p:.2e}')

conn.close()
print('Analytics workflow successfully executed.')